In [ ]:
import os
import torch
import torchaudio
import numpy as np
from tqdm import tqdm
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForSequenceClassification
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)

# ===============================================
# Path Setup
# ===============================================
WAV_PATH = "/home/alpaco/kmj/output.wav"  ## <-- 음성 입력

STEP1_DIR = "/home/alpaco/kmj/stutter_step1_best"
STEP2_DIR = "/home/alpaco/kmj/stutter_step2_best"

TARGET_SR = 16000
SEG_LEN = TARGET_SR * 3  # 3초 segment


# ======================================================
# Label Mapping (최종 확정)
# ======================================================
idx_to_name = {
    0: "SoundRep",
    1: "WordRep",
    2: "Prolongation",
    3: "Block"
}


# ======================================================
# Load Models
# ======================================================
proc1 = Wav2Vec2Processor.from_pretrained(STEP1_DIR)
model1 = Wav2Vec2ForSequenceClassification.from_pretrained(STEP1_DIR).to(device).eval()

proc2 = Wav2Vec2Processor.from_pretrained(STEP2_DIR)
model2 = Wav2Vec2ForSequenceClassification.from_pretrained(STEP2_DIR).to(device).eval()


# ======================================================
# Load Audio
# ======================================================
wav, sr = torchaudio.load(WAV_PATH)
wav = wav.mean(dim=0)  # mono

if sr != TARGET_SR:
    wav = torchaudio.functional.resample(wav, sr, TARGET_SR)

total_len = len(wav)
total_sec = total_len / TARGET_SR

print(f"\n==== Audio Info ====")
print(f"총 길이: {total_sec:.2f} sec")


# ======================================================
# Segment Split (3s + 마지막 padding)
# ======================================================
segments = []
num_segments = (total_len + SEG_LEN - 1) // SEG_LEN

for i in range(num_segments):
    s = i * SEG_LEN
    e = s + SEG_LEN
    seg = wav[s:e]

    if len(seg) < SEG_LEN:
        pad = torch.zeros(SEG_LEN - len(seg))
        seg = torch.cat([seg, pad])

    segments.append(seg)

print(f"총 segment 수: {num_segments}\n")


# ======================================================
# Inference
# ======================================================
step1_softmax = []
step1_pred = []
step2_pred = []

for i, seg in enumerate(tqdm(segments, desc="Inference")):

    # --------------------------
    # Step 1 prediction
    # --------------------------
    inp1 = proc1(seg.numpy(), sampling_rate=TARGET_SR,
                 return_tensors="pt").input_values.to(device)

    with torch.no_grad():
        out1 = model1(input_values=inp1)
        logits1 = out1.logits[0]
        probs1 = torch.softmax(logits1, dim=-1).cpu().numpy()

    pred1 = int(np.argmax(probs1))   # 0=normal, 1=stutter
    step1_softmax.append(probs1)
    step1_pred.append(pred1)

    # --------------------------
    # Step 2 prediction (only if stutter)
    # --------------------------
    if pred1 == 1:
        inp2 = proc2(seg.numpy(), sampling_rate=TARGET_SR,
                     return_tensors="pt").input_values.to(device)

        with torch.no_grad():
            out2 = model2(input_values=inp2)
            logits2 = out2.logits[0]

        pred2 = int(torch.argmax(logits2))  # 0~3
        step2_pred.append(pred2)
    else:
        step2_pred.append(None)


# ======================================================
# Summary
# ======================================================
total = len(step1_pred)
stutters = sum([1 for x in step1_pred if x == 1])

print("\n===== Summary =====")
print(f"총 segment: {total}")
print(f"말더듬 segment: {stutters}")
print(f"비율: {stutters}/{total} = {stutters/total:.3f}")
print("===================\n")


# ======================================================
# 상세 출력
# ======================================================
for i in range(total):
    p = step1_softmax[i]
    print(f"[SEG {i}]")

    print(f" Step1 softmax → normal:{p[0]:.3f} / stutter:{p[1]:.3f}")
    print(f" Step1 pred    → {step1_pred[i]}")

    if step1_pred[i] == 1:
        t = step2_pred[i]
        print(f" Step2 type    → {t} ({idx_to_name[t]})")

    print("---------------------------")


DEVICE: cuda

==== Audio Info ====
총 길이: 37.85 sec
총 segment 수: 13



Inference: 100%|██████████| 13/13 [00:00<00:00, 14.62it/s]


===== Summary =====
총 segment: 13
말더듬 segment: 8
비율: 8/13 = 0.615

[SEG 0]
 Step1 softmax → normal:0.356 / stutter:0.644
 Step1 pred    → 1
 Step2 type    → 1 (WordRep)
---------------------------
[SEG 1]
 Step1 softmax → normal:0.406 / stutter:0.594
 Step1 pred    → 1
 Step2 type    → 2 (Prolongation)
---------------------------
[SEG 2]
 Step1 softmax → normal:0.661 / stutter:0.339
 Step1 pred    → 0
---------------------------
[SEG 3]
 Step1 softmax → normal:0.869 / stutter:0.131
 Step1 pred    → 0
---------------------------
[SEG 4]
 Step1 softmax → normal:0.441 / stutter:0.559
 Step1 pred    → 1
 Step2 type    → 3 (Block)
---------------------------
[SEG 5]
 Step1 softmax → normal:0.396 / stutter:0.604
 Step1 pred    → 1
 Step2 type    → 2 (Prolongation)
---------------------------
[SEG 6]
 Step1 softmax → normal:0.293 / stutter:0.707
 Step1 pred    → 1
 Step2 type    → 2 (Prolongation)
---------------------------
[SEG 7]
 Step1 softmax → normal:0.916 / stutter:0.084
 Step1 pred  

In [5]:
import os
import torch
import torchaudio
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report, recall_score, f1_score
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)

MODEL_DIR = "/home/alpaco/kmj/stutter_step2_best"
VAL_CSV = "/home/alpaco/kmj/val_step2.csv"
WAV_ROOT = "/home/alpaco/kmj/all_wavs"

TARGET_SR = 16000
MAX_LEN = TARGET_SR * 3

# label4 → original label6
rev_map = {0:1, 1:2, 2:3, 3:4}

# ================================================
# Load CSV
# ================================================
df = pd.read_csv(VAL_CSV)

# ================================================
# Load Model
# ================================================
processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
model = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()


# ================================================
# Audio Preprocess (FAST)
# ================================================
def fix_audio(wav, sr):
    if wav.ndim == 2:
        wav = wav.mean(dim=0)
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    if len(wav) > MAX_LEN:
        wav = wav[:MAX_LEN]
    else:
        wav = torch.nn.functional.pad(wav, (0, MAX_LEN - len(wav)))
    return wav.float()


# ================================================
# FAST Loader (ThreadPool for WAV loading)
# ================================================
def load_one(fname):
    path = os.path.join(WAV_ROOT, fname)
    try:
        wav, sr = torchaudio.load(path, backend="soundfile")
        return fix_audio(wav, sr)
    except:
        return None


# ================================================
# GPU Inference
# ================================================
def predict_tensor(wav_tensor):
    inputs = processor(
        wav_tensor.numpy(),
        sampling_rate=TARGET_SR,
        return_tensors="pt"
    ).input_values.to(device)

    with torch.no_grad():
        logits = model(input_values=inputs).logits[0]
        pred = int(torch.argmax(logits))
    return pred


# ================================================
# 🔥 PROCESS: FAST MULTITHREADED + GPU STREAMING
# ================================================
preds = []
trues = []

fnames = df["filename"].tolist()
gt_labels = (df["label6"] - 1).tolist()  # convert 1~4 → 0~3

# thread pool for loading audio fast
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = [ex.submit(load_one, f) for f in fnames]

    for idx, fut in enumerate(tqdm(futures, desc="VALIDATION", unit="file")):
        wav_tensor = fut.result()

        if wav_tensor is None:
            preds.append(0)
            trues.append(gt_labels[idx])
            continue

        pred_label4 = predict_tensor(wav_tensor)

        preds.append(pred_label4)
        trues.append(gt_labels[idx])


# ================================================
# METRICS
# ================================================
cm = confusion_matrix(trues, preds)
macro_recall = recall_score(trues, preds, average="macro")
macro_f1 = f1_score(trues, preds, average="macro")
report = classification_report(trues, preds, digits=4)


# ================================================
# OUTPUT
# ================================================
print("\n===========================")
print("🔥 Confusion Matrix")
print("===========================")
print(cm)

print("\n===========================")
print("🔥 Classification Report")
print("===========================")
print(report)

print("\n🔥 Macro Recall:", round(macro_recall, 4))
print("🔥 Macro F1:", round(macro_f1, 4))


DEVICE: cuda


VALIDATION: 100%|██████████| 7890/7890 [04:46<00:00, 27.51file/s]


🔥 Confusion Matrix
[[ 432  356  340  412]
 [ 127  654  281  315]
 [ 137  503 1284  558]
 [ 287  559  666  979]]

🔥 Classification Report
              precision    recall  f1-score   support

           0     0.4395    0.2805    0.3424      1540
           1     0.3156    0.4749    0.3792      1377
           2     0.4994    0.5173    0.5082      2482
           3     0.4324    0.3930    0.4118      2491

    accuracy                         0.4245      7890
   macro avg     0.4217    0.4165    0.4104      7890
weighted avg     0.4345    0.4245    0.4229      7890


🔥 Macro Recall: 0.4165
🔥 Macro F1: 0.4104


In [ ]:
#step 2 학습코드
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import time
import torch
import torchaudio
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import recall_score, f1_score
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForSequenceClassification,
    get_cosine_schedule_with_warmup
)
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)


# ============================================
# Hyperparameters
# ============================================
MODEL_NAME = "kresnik/wav2vec2-large-xlsr-korean"

TARGET_SR = 16000
MAX_LEN = TARGET_SR * 3

BATCH = 64
LR = 2e-5
EPOCHS = 5
WARMUP_RATIO = 0.1
PATIENCE = 2

TRAIN_CSV = "/home/alpaco/kmj/train_step2.csv"
VAL_CSV   = "/home/alpaco/kmj/val_step2.csv"
WAV_ROOT  = "/home/alpaco/kmj/all_wavs"


# ============================================
# Load CSV
# ============================================
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

label_map = {1:0, 2:1, 3:2, 4:3}
train_df["label4"] = train_df["label6"].map(label_map).astype(int)
val_df["label4"]   = val_df["label6"].map(label_map).astype(int)

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)


# ============================================
# Audio fix
# ============================================
def fix_audio(wav, sr):
    if wav.ndim == 2:
        wav = wav.mean(dim=0)

    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)

    if len(wav) > MAX_LEN:
        wav = wav[:MAX_LEN]
    else:
        wav = torch.nn.functional.pad(wav, (0, MAX_LEN - len(wav)))

    return wav.float()


# ============================================
# Dataset
# ============================================
class StutterDataset(Dataset):
    def __init__(self, df, root):
        self.df = df
        self.root = root

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.root, row["filename"])

        try:
            wav, sr = torchaudio.load(path, backend="soundfile")
        except:
            return None

        wav = fix_audio(wav, sr)

        inputs = processor(
            wav.numpy(),
            sampling_rate=TARGET_SR,
            return_tensors="pt"
        )

        return {
            "input_values": inputs.input_values[0],
            "labels": torch.tensor(int(row["label4"]), dtype=torch.long),
        }


# ============================================
# Collate (빈배치 처리)
# ============================================
def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None

    input_values = torch.stack([b["input_values"] for b in batch])
    labels = torch.stack([b["labels"] for b in batch])
    return {"input_values": input_values, "labels": labels}


train_loader = DataLoader(
    StutterDataset(train_df, WAV_ROOT),
    batch_size=BATCH, shuffle=True, num_workers=4, collate_fn=collate_fn
)

val_loader = DataLoader(
    StutterDataset(val_df, WAV_ROOT),
    batch_size=BATCH, shuffle=False, num_workers=4, collate_fn=collate_fn
)


# ============================================
# Class Weights (normalize 제거)
# ============================================
counts = train_df["label4"].value_counts().sort_index().values
weights = 1 / torch.tensor(counts, dtype=torch.float)
weights = weights.to(device)
print("Class Weights:", weights)


# ============================================
# Focal Loss
# ============================================
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=None, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = torch.nn.CrossEntropyLoss(reduction='none')

    def forward(self, logits, targets):
        ce = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        pt = probs[torch.arange(len(targets)), targets]

        focal = (1 - pt) ** self.gamma * ce
        if self.alpha is not None:
            focal = self.alpha[targets] * focal
        return focal.mean()

loss_fn = FocalLoss(alpha=weights, gamma=2)


# ============================================
# Model Load + Partial Unfreeze (21~23)
# ============================================
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4
).to(device)

for p in model.wav2vec2.parameters():
    p.requires_grad = False

for name, p in model.wav2vec2.named_parameters():
    if any(x in name for x in ["encoder.layers.21", "encoder.layers.22", "encoder.layers.23"]):
        p.requires_grad = True

for p in model.projector.parameters():
    p.requires_grad = True
for p in model.classifier.parameters():
    p.requires_grad = True


optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_cosine_schedule_with_warmup(
    optimizer, warmup_steps, total_steps
)


# ============================================
# Training
# ============================================
scaler = torch.cuda.amp.GradScaler()
best_f1 = 0
no_improve = 0

total_start = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()
    model.train()
    train_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for i, batch in enumerate(pbar):
        if batch is None:
            continue

        optimizer.zero_grad()

        inputs = batch["input_values"].to(device)
        labels = batch["labels"].to(device)

        with torch.cuda.amp.autocast():
            outputs = model(input_values=inputs)
            loss = loss_fn(outputs.logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss += loss.item()

        progress = (i + 1) / len(train_loader)
        elapsed = time.time() - epoch_start
        eta = elapsed / progress - elapsed

        pbar.set_postfix({"loss": f"{loss.item():.4f}",
                          "ETA": f"{eta/60:.1f} min"})

    # -----------------------------
    # Validation
    # -----------------------------
    model.eval()
    val_loss = 0
    preds, trues = [], []

    with torch.no_grad():
        for batch in val_loader:
            if batch is None:
                continue

            inputs = batch["input_values"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_values=inputs)
            loss = loss_fn(outputs.logits, labels)

            val_loss += loss.item()
            pred = torch.softmax(outputs.logits, dim=1).argmax(dim=1)

            preds.extend(pred.cpu().numpy())
            trues.extend(labels.cpu().numpy())

    macro = recall_score(trues, preds, average="macro")
    macro_f1 = f1_score(trues, preds, average="macro")

    print(f"\n=== Epoch {epoch+1} ===")
    print(f"Train Loss: {train_loss/len(train_loader):.4f}")
    print(f"Val Loss  : {val_loss/len(val_loader):.4f}")
    print(f"Macro Rec.: {macro:.4f}")
    print(f"Macro F1  : {macro_f1:.4f}")

    # -----------------------------
    # Early Stopping
    # -----------------------------
    if macro_f1 > best_f1:
        best_f1 = macro_f1
        no_improve = 0

        model.save_pretrained("/home/alpaco/kmj/stutter_step2_best")
        processor.save_pretrained("/home/alpaco/kmj/stutter_step2_best")
        print("🔥 BEST MODEL SAVED!")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("\n🛑 Early Stopping Triggered!")
            break

total_time = (time.time() - total_start) / 60
print(f"\nTotal Training Time: {total_time:.2f} min")
print("Step2 Training Finished.")


DEVICE: cuda
Class Weights: tensor([8.1149e-05, 9.0794e-05, 5.0363e-05, 5.0176e-05], device='cuda:0')


Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at kresnik/wav2vec2-large-xlsr-korean and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_828754/220677379.py:197: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/5:   0%|          | 0/987 [00:00<?, ?it/s]/tmp/ipykernel_828754/220677379.py:219: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/5:   0%|          | 0/987 [00:00<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
## step1 학습코드
# ============================================================
# GPU 고정
# ============================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import time
import torch
import torchaudio
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import recall_score, f1_score
from tqdm import tqdm

from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForSequenceClassification,
    get_cosine_schedule_with_warmup
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)


# ============================================================
# Hyperparameters
# ============================================================
MODEL_NAME = "kresnik/wav2vec2-large-xlsr-korean"
TARGET_SR = 16000
MAX_LEN = TARGET_SR * 3

BATCH = 32
LR = 2e-5
EPOCHS = 5
WARMUP_RATIO = 0.1
PATIENCE = 2

root_wav = "/home/alpaco/kmj/all_wavs"
train_csv = "/home/alpaco/kmj/train.csv"
val_csv   = "/home/alpaco/kmj/val.csv"

# ============================================================
# Load Data
# ============================================================
train_df = pd.read_csv(train_csv)
val_df   = pd.read_csv(val_csv)

# Binary label 생성 (0-normal / 1-stutter)
train_df["binary"] = train_df["label6"].apply(lambda x: 0 if x == 0 else 1)
val_df["binary"]   = val_df["label6"].apply(lambda x: 0 if x == 0 else 1)

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)


# ============================================================
# Dataset
# ============================================================
def fix_audio(wav, sr):
    if wav.ndim == 2:
        wav = wav.mean(dim=0)

    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)

    if len(wav) > MAX_LEN:
        wav = wav[:MAX_LEN]
    else:
        wav = torch.nn.functional.pad(wav, (0, MAX_LEN - len(wav)))

    return wav.float()


class BinaryDataset(Dataset):
    def __init__(self, df, root):
        self.df = df
        self.root = root

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        wav, sr = torchaudio.load(
            os.path.join(self.root, row["filename"]),
            backend="soundfile"
        )
        wav = fix_audio(wav, sr)

        inputs = processor(
            wav.numpy(),
            sampling_rate=TARGET_SR,
            return_tensors="pt"
        )

        return {
            "input_values": inputs.input_values[0],
            "labels": torch.tensor(row["binary"], dtype=torch.long),
        }


train_loader = DataLoader(BinaryDataset(train_df, root_wav),
                          batch_size=BATCH, shuffle=True, num_workers=4)
val_loader = DataLoader(BinaryDataset(val_df, root_wav),
                        batch_size=BATCH, shuffle=False, num_workers=4)


# ============================================================
# Class Weights
# ============================================================
counts = train_df["binary"].value_counts().sort_index().values
weights = 1 / counts
weights = weights / weights.sum()
weights = torch.tensor(weights, dtype=torch.float).to(device)
print("Class Weights:", weights)


# ============================================================
# Focal Loss
# ============================================================
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=None, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = torch.nn.CrossEntropyLoss(reduction='none')

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        pt = probs[torch.arange(len(targets)), targets]
        focal = (1 - pt) ** self.gamma * ce_loss

        if self.alpha is not None:
            focal = self.alpha[targets] * focal

        return focal.mean()


loss_fn = FocalLoss(alpha=weights, gamma=2)


# ============================================================
# Model Load (Partial Unfreeze: 21, 22, 23)
# ============================================================
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
).to(device)

# 1) freeze 전체 encoder
for param in model.wav2vec2.parameters():
    param.requires_grad = False

# 2) encoder 마지막 레이어 21,22,23만 unfreeze
for name, param in model.wav2vec2.named_parameters():
    if any(layer in name for layer in [
        "encoder.layers.21",
        "encoder.layers.22",
        "encoder.layers.23"
    ]):
        param.requires_grad = True

# 3) head 전체 train
for param in model.projector.parameters():
    param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True


optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_cosine_schedule_with_warmup(
    optimizer, warmup_steps, total_steps
)


# ============================================================
# Training Loop
# ============================================================
scaler = torch.cuda.amp.GradScaler()
best_macro = 0
no_improve = 0

total_start = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()

    model.train()
    train_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", ncols=120)

    for batch in pbar:
        optimizer.zero_grad()

        inputs = batch["input_values"].to(device)
        labels = batch["labels"].to(device)

        with torch.cuda.amp.autocast():
            outputs = model(inputs)
            loss = loss_fn(outputs.logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})


    # ---------------- Validation ----------------
    model.eval()
    val_loss = 0
    preds = []
    trues = []

    with torch.no_grad():
        for batch in val_loader:
            inputs = batch["input_values"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(inputs)
            loss = loss_fn(outputs.logits, labels)

            val_loss += loss.item()

            prob = torch.softmax(outputs.logits, dim=1)
            pred = prob.argmax(dim=1)

            preds.extend(pred.cpu().numpy())
            trues.extend(labels.cpu().numpy())

    macro = recall_score(trues, preds, average="macro")
    macro_f1 = f1_score(trues, preds, average="macro")

    epoch_time = time.time() - epoch_start
    remain = (EPOCHS - epoch - 1) * epoch_time

    print(f"\n=== Epoch {epoch+1} Results ===")
    print(f"Train Loss    : {train_loss/len(train_loader):.4f}")
    print(f"Val Loss      : {val_loss/len(val_loader):.4f}")
    print(f"Macro Recall  : {macro:.4f}")
    print(f"Macro F1      : {macro_f1:.4f}")
    print(f"Epoch Time    : {epoch_time/60:.2f} min")
    print(f"Remaining ETA : {remain/60:.2f} min\n")

    # ---------------- Early Stopping ----------------
    if macro > best_macro:
        best_macro = macro
        no_improve = 0

        model.save_pretrained("/home/alpaco/kmj/stutter_step1_best")
        processor.save_pretrained("/home/alpaco/kmj/stutter_step1_best")
        print("🔥 BEST MODEL SAVED!")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("\n🛑 Early stopping triggered!")
            break


total_time = (time.time() - total_start) / 60
print(f"\nTotal Training Time: {total_time:.2f} min")
print("🔥 Step1 Training Finished.")
